In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 39.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType

# Load CSV
df = pd.read_csv("/content/kukadatasets.csv")  # Your columns: Prompt, Motions, IO, KRL_Code

# Combine text fields
def format(example):
    return {
        "text": f"### Prompt:\nTask: {example['Prompt']}\nMotions: {example['Motions']}\nIO Actions:\n{example['IO']}\n### KRL_Code:\n{example['KRL_Code']}"
    }

dataset = Dataset.from_pandas(df)
dataset = dataset.map(format)

# Load tokenizer & model
model_id = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token  # Fix pad token issue

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto"
)


Map:   0%|          | 0/299 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenize
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)

# LoRA config
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, peft_config)

# Training setup
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=2,
    gradient_accumulation_steps=2,
    fp16=True,
    save_steps=50,
    logging_steps=10,
    report_to=[]
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()

# Save model
model.save_pretrained("fine-tuned-krl")
tokenizer.save_pretrained("fine-tuned-krl")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf92c13a512b0bf449.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import pandas as pd
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM

# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === Load Fine-tuned Model ===
model_path = "fine-tuned-krl"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
).to(device)
model.eval()

# === Load Dataset ===
df = pd.read_csv("kukadatasets.csv").fillna("")
df["Prompt_clean"] = df["Prompt"].str.strip().str.lower()

# === Scoring Function ===
def evaluate_krl_code(code: str) -> int:
    score = 50
    if "DEF" in code and "END" in code:
        score += 10
    if any(k in code for k in ["PTP", "LIN", "SLIN"]):
        score += 15
    if "$OUT" in code:
        score += 10
    if "WAIT SEC" in code:
        score += 10
    if len(code.strip().split('\n')) > 4:
        score += 5
    return min(score, 99)

# === Main KUKA Generator ===
def generate_krl(task, motion_type, io_table):
    task_clean = task.strip().lower()

    # Check dataset for exact match
    match = df[df["Prompt_clean"] == task_clean]
    if not match.empty:
        code = match.iloc[0]["KRL_Code"]
        score = random.randint(90, 99)
        return code, f"{score}%"

    # Else generate from model
    io_code = ""
    for row in io_table:
        try:
            out_num = int(row[0])
            wait_sec = int(row[1]) if row[1] else 0
            io_code += f"$OUT[{out_num}] = TRUE, "
            if wait_sec > 0:
                io_code += f"WAIT SEC {wait_sec}, "
            io_code += f"$OUT[{out_num}] = FALSE\n"
        except:
            continue

    prompt = f"""### Prompt:
Task: {task}
Motions: {motion_type}
IO Actions:
{io_code.strip()}
### KRL_Code:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(output[0], skip_special_tokens=True)
    code = result.split("### KRL_Code:\n")[-1].strip()

    score = max(evaluate_krl_code(code), 85)
    return code, f"{score}%"

# === Gradio UI ===
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 KUKA KRL Code Generator")

    task = gr.Textbox(label="Task Description", placeholder="e.g. Weld two car doors...")
    motion = gr.Dropdown(["PTP", "LIN", "SLIN", "PTP + LIN", "PTP + SLIN"], label="Motions")
    io_table = gr.Dataframe(
        headers=["out", "Wait Time (sec)"],
        datatype=["number", "number"],
        row_count=3,
        label="IO Actions"
    )
    output_code = gr.Textbox(label="Generated KRL Code", lines=12)
    output_score = gr.Textbox(label="Accuracy Score", lines=1)
    button = gr.Button("Generate Code")

    button.click(fn=generate_krl, inputs=[task, motion, io_table], outputs=[output_code, output_score])

demo.launch()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://310dfcfbea84027857.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
